# Father — post-mortem investigation

**Question:** what activity can disk, timeline and RAM reconstruct, and what does
combining them add?

[Investigation rules](../../ai/RULES.md)


## Evidence and scope

This notebook examines the preserved disk and memory evidence for one selected
Father run. The investigation is scenario-informed: execution records define
expected effects, while forensic conclusions require acquired evidence.


In [2]:
import json
import sys
from pathlib import Path

RUN_ID = "father-u22-20260913-01"
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "shared" / "experiments").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if not (PROJECT_ROOT / "shared" / "experiments").is_dir():
    raise FileNotFoundError("Start the notebook in the repository root or investigations/father")

sys.path.insert(0, str(PROJECT_ROOT / "investigations" / "father"))

from investigation_utils import run_command

RUN_ROOT = PROJECT_ROOT / "shared" / "experiments" / RUN_ID
INVESTIGATION_ROOT = RUN_ROOT / "investigation"
MANIFEST_PATH = RUN_ROOT / "manifest.json"
ACQUISITION_PATH = RUN_ROOT / "dumps" / "acquisition.json"
COMMAND_LOG_PATH = RUN_ROOT / "command_log.jsonl"
DATA_DIR = INVESTIGATION_ROOT / "data"
OUT_DIR = INVESTIGATION_ROOT / "output"
RECOVERED_DIR = INVESTIGATION_ROOT / "recovered"
FINDINGS_DIR = INVESTIGATION_ROOT / "findings"
RESULTS_DIR = INVESTIGATION_ROOT / "results"

In [3]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
acquisition = json.loads(ACQUISITION_PATH.read_text(encoding="utf-8"))
if manifest["run_id"] != RUN_ID:
    raise ValueError("Manifest run ID does not match RUN_ID")

claims = manifest["claims"]
DISK_IMAGE = RUN_ROOT / "dumps" / acquisition["disk"]["path"]
DISK_SEGMENTS = [RUN_ROOT / "dumps" / path for path in acquisition["disk"]["segments"]]
MEMORY_IMAGE = RUN_ROOT / "dumps" / acquisition["memory"]["path"]


In [4]:
from IPython.display import Markdown, display

platform = manifest["platform"]
case_rows = [
    ("Run ID", RUN_ID),
    ("Scenario", manifest["scenario"]),
    ("Guest distribution and version", platform["guest_os"]),
    ("Kernel", platform["kernel"]),
    ("Architecture", platform["arch"]),
    ("Timezone", platform["timezone"])
]
display(Markdown("| Case | Recorded value |\n|---|---|\n" +
                 "\n".join(f"| {name} | {value} |" for name, value in case_rows)))

evidence_rows = []
for name, image, record in (
    ("Disk", DISK_IMAGE, acquisition["disk"]),
    ("RAM", MEMORY_IMAGE, acquisition["memory"]),
):
    interval = f'{record["started_at"]} to {record["ended_at"]}'
    evidence_rows.append(
        f'| {name} | {image.relative_to(RUN_ROOT)} | {record["size_bytes"]:,} bytes | '
        f'{record["sha256"]} | {interval} |'
    )
display(Markdown(
    "| Evidence | Relative path | Recorded size | Recorded SHA-256 | Acquisition interval |\n"
    "|---|---|---:|---|---|\n" + "\n".join(evidence_rows)
))
print("Disk hash scope: recorded logical disk stream, not an E01 segment.")
print("RAM hash scope: recorded memory dump file.")

scenario_rows = [
    ("Scenario", manifest["scenario"]),
    ("Status", manifest["status"]),
    ("Started at", manifest["timestamps"]["scenario_started_at"]),
    ("Ended at", manifest["timestamps"]["scenario_ended_at"])]

display(Markdown(
    "| Scenario info | Value |\n"
    "|---|---|\n" +
    "\n".join(f"| {name} | {value} |" for name, value in scenario_rows)))

| Case | Recorded value |
|---|---|
| Run ID | father-u22-20260913-01 |
| Scenario | father |
| Guest distribution and version | Ubuntu 22.04.5 LTS |
| Kernel | 5.15.0-179-generic |
| Architecture | x86_64 |
| Timezone | Etc/UTC |

| Evidence | Relative path | Recorded size | Recorded SHA-256 | Acquisition interval |
|---|---|---:|---|---|
| Disk | dumps/disk/evidence_disk.E01 | 10,737,418,240 bytes | ae591cfdfb26569b16478bbbc80bdd4e7f9bcc1741a84b13ebb9cb8bdcb65aec | 2026-09-13T20:46:38.398053Z to 2026-09-13T20:47:16.034097Z |
| RAM | dumps/memory/mem.raw | 2,147,747,795 bytes | e4089156b81a250dda36391932bf8011f70d0b54cd7acd4fcc859758d12472e9 | 2026-09-13T20:46:30.996505Z to 2026-09-13T20:46:32.853530Z |

Disk hash scope: recorded logical disk stream, not an E01 segment.
RAM hash scope: recorded memory dump file.


| Scenario info | Value |
|---|---|
| Scenario | father |
| Status | completed |
| Started at | 2026-09-13T20:44:59.986582Z |
| Ended at | 2026-09-13T20:46:30.976360Z |

In [5]:
if DISK_IMAGE not in DISK_SEGMENTS:
    raise ValueError("Disk entry point is absent from the declared segments")

required_paths = [
    MANIFEST_PATH, ACQUISITION_PATH, COMMAND_LOG_PATH,
    *DISK_SEGMENTS, MEMORY_IMAGE,
    RUN_ROOT / manifest["artifacts"]["acquisition_log"],
    RUN_ROOT / manifest["artifacts"]["terminal_transcript"],
]
for item in manifest["inputs"]:
    required_paths.append(RUN_ROOT / item["build_json"]["path"])
    required_paths.extend(RUN_ROOT / artifact["path"] for artifact in item["artifacts"])

missing = [path for path in dict.fromkeys(required_paths) if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required evidence: " + ", ".join(str(path) for path in missing))
print("required evidence present")

for directory in (DATA_DIR, FINDINGS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


required evidence present


### Scope limitations

- The manifest and command log are execution-derived references, not forensic findings; this investigation is scenario-informed rather than blind.
- Disk and RAM were acquired at different times. Acquisition times are not attacker-event times or a continuous snapshot.
- The hashes above are recorded acquisition metadata, not fresh verification results.
- The recorded source working tree was modified, limiting exact source reproducibility.
- The guest has a vanilla profile with limited logging.


## 1. Disk evidence examination

### Block 1: Image verification and partition layout
**Question:** Does the acquired disk image match recorded acquisition hashes, and what partition offset hosts the root filesystem?

In [40]:
VERIFY = False # avoid running ewfverify for debugging purposes

disk_info = run_command(f'ewfinfo "{DISK_IMAGE}"', label="s1-01-ewfinfo", out_dir=OUT_DIR)

if (not VERIFY) : print("Verification skipped... (VERIFY = False)")

!ewfverify -V {help}
if(VERIFY):
    disk_verification = run_command(
        f'ewfverify -j $(( $(nproc) - 2 )) -d sha256 "{DISK_IMAGE}"',
        label="s1-02-ewfverify",
        out_dir=OUT_DIR, check=True
    )
    manifest_sha256 = acquisition['disk']['sha256']

    ewf_verify_out = OUT_DIR/"s1-02-ewfverify.txt"

    ewf_calculated_sha256 = run_command(
        f"""grep "SHA256 hash calculated" "{ewf_verify_out}" | awk '{{print $NF}}'"""
    , verbose = False).stdout.strip()

    if manifest_sha256 == ewf_calculated_sha256:
        print(f"[SUCCESS] Hashes match: {ewf_calculated_sha256}")
    else:
        raise AssertionError(
        f"[FAILED] Hash mismatch!\n  Expected:   {manifest_sha256}\n  Calculated: {ewf_calculated_sha256}"
        )

$ ewfinfo "/home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01"
ewfinfo 20240506

Acquiry information:
	Acquisition date:	Sun Sep 13 22:46:40 2026
	System date:		Sun Sep 13 22:46:40 2026
	Operating system used:	Linux
	Software version used:	20240506
	Password:		N/A

EWF information:
	File format:		EnCase 6
	Sectors per chunk:	64
	Error granularity:	64
	Compression method:	deflate
	Compression level:	no compression

Media information:
	Media type:		fixed disk
	Is physical:		yes
	Bytes per sector:	512
	Number of sectors:	20971520
	Media size:		10 GiB (10737418240 bytes)

Digest hash information:
	MD5:			357e7bf61252249b33f48c282ee951ef

Verification skipped... (VERIFY = False)
zsh:1: parse error near `help,'


In [ ]:
from investigation_utils import get_rootfs_offset

# Print partition table in order to identify the correct root partition
partition_layout = run_command(f"mmls {DISK_IMAGE}", label="s1-03-mmls", out_dir=OUT_DIR)

# Find offset sector of the root partition (Ext4)
offset_sector = get_rootfs_offset(DISK_IMAGE)
assert offset_sector is not None, f"Could not find an Ext4 partition in {DISK_IMAGE}"
offset_bytes = offset_sector * 512

# Inspect the Ext4 Filesystem metadata
fs_metadata = run_command(["fsstat", "-o", offset_sector, DISK_IMAGE], label="s1-04-fsstat", out_dir=OUT_DIR, verbose=False)
print("\n".join(fs_metadata.stdout.splitlines()[:25]))

# Print default time zone for the machine in order to align investigation
time_inode = run_command(f"ifind -o {offset_sector} -n /etc/localtime {DISK_IMAGE}").stdout.strip()
timezone = run_command(f"istat -o {offset_sector} {DISK_IMAGE} {time_inode} | grep \"symbolic\"", verbose=False).stdout.split(":")[-1]

print(f"HOST TIMEZONE: {timezone}")


## Create body-file via fls -r and print first 25 lines
# body_file = run_command(f"fls -r -o {offset_sector} {DISK_IMAGE}", label="bodyfile", out_dir=DATA_DIR, verbose=False)
# print("\n".join(body_file.stdout.splitlines()[:25]))

$ mmls /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
GUID Partition Table (EFI)
Offset Sector: 0
Units are in 512-byte sectors

      Slot      Start        End          Length       Description
000:  Meta      0000000000   0000000000   0000000001   Safety Table
001:  -------   0000000000   0000002047   0000002048   Unallocated
002:  Meta      0000000001   0000000001   0000000001   GPT Header
003:  Meta      0000000002   0000000033   0000000032   Partition Table
004:  013       0000002048   0000010239   0000008192   
005:  014       0000010240   0000227327   0000217088   
006:  000       0000227328   0020971486   0020744159   
007:  -------   0020971487   0020971519   0000000033   Unallocated
$ fsstat -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
FILE SYSTEM INFORMATION
--------------------------------------------
File System Type: Ext4
Volume Name: cl

In [38]:
!who

anto     tty7         2026-09-18 11:15 (:0)


### Output interpretation
From the code above we have:
- verify the acquisition via `ewfinfo` and `ewfverify` commands
- identify the correct partition offset via `mmls` and `fsstat`
- create a simple body-file from the selected partition

### Block 2: Preload configuration analysis
We assume we have the suspecion that an LD_PRELOAD rootkit infected the system we're analyzing:

**Question:** Does `/etc/ld.so.preload` exist, and what shared object does it reference?

In [8]:
# 1. Find the inode number of /etc/ld.so.preload
preload_lookup = run_command(
    ["ifind", "-o", offset_sector, "-n", "/etc/ld.so.preload", DISK_IMAGE], 
    label="s2-01-preload-ifind", out_dir=OUT_DIR
)
preload_inode = preload_lookup.stdout.strip()
print(f"/etc/ld.so.preload inode: {preload_inode}")

# 2. Print inode metadata info to check allocation status and timestamp
run_command(
    ["istat", "-o", offset_sector, "-z", "UTC", DISK_IMAGE, preload_inode],
    label="s2-02-preload-istat",
    out_dir=OUT_DIR,
)

# 3. Print the content of the preload file
icat_cmd = run_command(["icat","-o",offset_sector,DISK_IMAGE,preload_inode],
    label="s2-03-preload-icat",
    out_dir=OUT_DIR,
)

ld_preload_lib = icat_cmd.stdout.strip()

print(f"\nSuspicious lib: {ld_preload_lib}")

$ ifind -o 227328 -n /etc/ld.so.preload /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
74252
/etc/ld.so.preload inode: 74252
$ istat -o 227328 -z UTC /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 74252
inode: 74252
Allocated
Group: 4
Generation Id: 3702835834
uid / gid: 0 / 0
mode: rrw-r--r--
Flags: Extents, 
size: 17
num of links: 1

Inode Times:
Accessed:	2026-09-13 20:46:32.884000000 (UTC)
File Modified:	2026-09-13 20:46:32.880000000 (UTC)
Inode Modified:	2026-09-13 20:46:32.880000000 (UTC)
File Created:	2026-09-13 20:46:32.880000000 (UTC)

Direct Blocks:
296191 
$ icat -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 74252
/lib/selinux.so.3
Suspicious lib: /lib/selinux.so.3


### Output Interpretation
- The file `/etc/ld.so.preload` has been found and its inode is 74252
- The inode MAC times are compatible with the scenario execution one, last access possibly happened during shutdown
- The content of the file references a suspicious library `/lib/selinux.so.3` that shouldn't be present (moreover ubuntu usually utilizes AppArmor and not SeLinux)

### Block 3: Suspicious library inspection and analysis
**Question:** What are the static properties (type, hash, strings, symbols) of the referenced shared object?

In [9]:
# 1. Find the inode number of the suspcious library
sus_lib_lookup = run_command(
    ["ifind", "-o", offset_sector, "-n", ld_preload_lib, DISK_IMAGE],
    label="s3-01-sus_lib-ifind",
    out_dir=OUT_DIR,
    check=False
)

# 2. TSK doesn't resolve symbolic link automatically.. it should be done manually
resolved_path = ""
if sus_lib_lookup.returncode != 0:
    lib_inode = run_command(f"ifind -o {offset_sector} -n lib {DISK_IMAGE}").stdout.strip()
    sym_link = run_command(f"istat -o {offset_sector} {DISK_IMAGE} {lib_inode} | grep  \"symbolic\"").stdout.strip().split(' ')[-1]

    # 3. lib -> usr/lib, let's retry ifind with the full path now
    sym_prepend = "".join(sym_link.split("/")[:-1])
    resolved_path = sym_prepend + ld_preload_lib
    sus_lib_lookup = run_command(
        ["ifind", "-o", offset_sector, "-n", sym_prepend + ld_preload_lib, DISK_IMAGE],
        label="s3-02-sus_lib-ifind",
        out_dir=OUT_DIR,
        check=False,
    )

sus_lib_inode = sus_lib_lookup.stdout.strip()
print(f"Inode of the lib: {ld_preload_lib} ({resolved_path}): {sus_lib_inode}")

# 4. The inode has been identified, let's inspect it
run_command(f"istat -o {offset_sector} -z UTC {DISK_IMAGE} {sus_lib_inode}", out_dir=OUT_DIR, label="s3-03-sus-lib-istat")

# 5. And let's extract the content for basic static characterization
sus_lib_label = f"{ld_preload_lib.split('/')[-1]}-extracted.bin"
run_command(f"icat -o {offset_sector} {DISK_IMAGE} {sus_lib_inode}", verbose=False, out_dir=DATA_DIR, label=sus_lib_label, binary=True)

print(f"Extracted suspicious library at {DATA_DIR/sus_lib_label} ")
extracted_sus_lib = DATA_DIR/sus_lib_label

# 6. Basic static characterization and analysis
run_command(f"file {extracted_sus_lib}", out_dir=OUT_DIR, label="s3-04-sus-lib-file")
run_command(f"sha256sum {extracted_sus_lib}", out_dir=DATA_DIR, label="s3-04-sus-lib-sha256")
run_command(f"strings {extracted_sus_lib}", out_dir=DATA_DIR, label="s3-05-sus-lib-strings")

$ ifind -o 227328 -n /lib/selinux.so.3 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
Error extracting file from image (ext2fs_dir_open_meta: Error reading directory contents: 1542
)
$ ifind -o 227328 -n lib /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
1542
$ istat -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 1542 | grep  "symbolic"
symbolic link to: usr/lib
$ ifind -o 227328 -n usr/lib/selinux.so.3 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
74253
Inode of the lib: /lib/selinux.so.3 (usr/lib/selinux.so.3): 74253
$ istat -o 227328 -z UTC /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 74253
inode: 74253
Allocated
Group: 4
Generation Id: 33261919
uid / gid: 0 / 0
mod

CompletedProcess(args='strings /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/data/selinux.so.3-extracted.bin', returncode=0, stdout='XCgY\n@)^(Hf\t\n__gmon_start__\n_ITM_deregisterTMCloneTable\n_ITM_registerTMCloneTable\n__cxa_finalize\no_accept\ndlsym\ngetegid\nntohs\nfork\nread\nstrstr\nmemfrob\ngeteuid\nsetgid\ndup2\nexecl\n__errno_location\n__stack_chk_fail\no_access\nlpe_drop_shell\nmemset\n__lxstat\no_execve\nwait\nfwrite\nfclose\nexit\ntimebomb\ntime\ngetenv\nsetuid\nseteuid\nunsetenv\nbackconnect\ninet_addr\nhtons\nsocket\nfalsify_tcp\ntmpfile\nfputs\nfgets\nrewind\no_verify\ngcry_pk_verify\nstrfry\no_open\no_open64\n__lxstat64\no_openat\nfstatat\no_opendir\no_fopen\nstrncmp\no_fopen64\no_readdir\no_lxstat\no_lxstat64\no_lstat\no_fstat\no_unlink\no_unlinkat\npampassword\nexfil\nfprintf\noldconv\nnewconv\nfree\nstrdup\no_pam_authenticate\npam_get_item\nstrcmp\nlibc.so.6\nGLIBC_2.33\nGLIBC_2.34\nGLIBC_2.4\nGLIBC_2.2.5\nu+UH\n@ =9\n@

### Output interpretation
- The suspicious library has been found via 'manual resolve' of the symbolic link (`lib/` was actually a link to `usr/lib`)
- From the istat output we can notice that MAC times seems aligned with scenario ones but File Modified is actually suspicious (its timestamp is not aligned 2026-01-30 out the scenario window) and could be a Timestomp attempts
- From the extracted strings we can extract useful 'suspicious' strings that can help understand the behaviour of the rootkit 

### Block 4: Library internal timestamp anomaly analysis
**Question:** Do the library's filesystem timestamps exhibit internal anomalies?

In [10]:
library_metadata = run_command(
    ["istat", "-z", "UTC", "-o", offset_sector, DISK_IMAGE, sus_lib_inode],
    label="s1-16-library-istat.txt", out_dir=DATA_DIR
)
print("Target Library timestamps:")
for line in library_metadata.stdout.splitlines():
    if "File Modified" in line or "Inode Modified" in line or "File Created" in line:
        print(line.strip())


$ istat -z UTC -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 74253
inode: 74253
Allocated
Group: 4
Generation Id: 33261919
uid / gid: 0 / 0
mode: rrw-r--r--
Flags: Extents, 
size: 32784
num of links: 1

Inode Times:
Accessed:	2026-09-13 20:46:00.416000000 (UTC)
File Modified:	2026-01-30 08:20:56.000000000 (UTC)
Inode Modified:	2026-09-13 20:45:24.340000000 (UTC)
File Created:	2026-09-13 20:45:24.328000000 (UTC)

Direct Blocks:
338478 338479 338480 338481 338482 338483 338484 338485 
338486 
Target Library timestamps:
File Modified:	2026-01-30 08:20:56.000000000 (UTC)
Inode Modified:	2026-09-13 20:45:24.340000000 (UTC)
File Created:	2026-09-13 20:45:24.328000000 (UTC)


**Interpretation:** The recovered library has a modification time that predates its acquired inode-change and creation times. This is a suspicious relationship consistent with possible timestamp manipulation, but it does not establish the operation or its actor. Timeline correlation may provide additional or conflicting context.
### Block 5: Correlation between `ctime` and login session
**Question:** Is it possible to find a login session related to that timestamp? 

In [11]:
# Extract lastlog
run_command(f"fcat -o {offset_sector} /var/log/wtmp {DISK_IMAGE}", out_dir=DATA_DIR,label="wtmp.bin", binary=True)

# Read it with last
run_command(f"TZ=UTC last -f {DATA_DIR/"wtmp.bin"}")

$ fcat -o 227328 /var/log/wtmp /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
$ TZ=UTC last -f /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/data/wtmp.bin
labuser  pts/0        192.168.100.1    Sun Sep 13 20:44 - 20:46  (00:01)
reboot   system boot  5.15.0-179-gener Sun Sep 13 20:44 - 20:46  (00:01)
labuser  pts/0        192.168.100.1    Tue Aug 11 17:13 - 17:13  (00:00)
labuser  pts/0        192.168.100.1    Tue Aug 11 17:13 - 17:13  (00:00)
labuser  pts/0        192.168.100.1    Tue Aug 11 17:13 - 17:13  (00:00)
reboot   system boot  5.15.0-179-gener Tue Aug 11 17:13 - 17:14  (00:00)

wtmp.bin begins Tue Aug 11 17:13:43 2026


CompletedProcess(args='TZ=UTC last -f /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/data/wtmp.bin', returncode=0, stdout='labuser  pts/0        192.168.100.1    Sun Sep 13 20:44 - 20:46  (00:01)\nreboot   system boot  5.15.0-179-gener Sun Sep 13 20:44 - 20:46  (00:01)\nlabuser  pts/0        192.168.100.1    Tue Aug 11 17:13 - 17:13  (00:00)\nlabuser  pts/0        192.168.100.1    Tue Aug 11 17:13 - 17:13  (00:00)\nlabuser  pts/0        192.168.100.1    Tue Aug 11 17:13 - 17:13  (00:00)\nreboot   system boot  5.15.0-179-gener Tue Aug 11 17:13 - 17:14  (00:00)\n\nwtmp.bin begins Tue Aug 11 17:13:43 2026\n')

The last login entry shows `labuser  pts/0        192.168.100.1    Sun Sep 13 20:44 - 20:46 UTC`. `ctime` is included in that time window

## Block 6: Check for history files

In [37]:
# Grep on bodyfile for shell history files
run_command(f"grep -iE '\\.(bash|zsh|sh)_history' {DATA_DIR/"bodyfile.txt"}",check=False)

# Directory Listing for labuser and root dir
run_command(
    f"grep -iE '.*d\\/d.*[[:space:]]+(labuser|root)[[:space:]]*' {DATA_DIR/"bodyfile.txt"}",
    check=False,)

print("labuser listing")
run_command(f"fls -o {offset_sector} {DISK_IMAGE} 258049")

print("Root dir listing")
run_command(f"fls -o {offset_sector} {DISK_IMAGE} 1550")

$ grep -iE '\.(bash|zsh|sh)_history' /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/data/bodyfile.txt
$ grep -iE '.*d\/d.*[[:space:]]+(labuser|root)[[:space:]]*' /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/data/bodyfile.txt
+ d/d 258049:	labuser
d/d 1550:	root


labuser listing
$ fls -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 258049
r/r 258050:	.bash_logout
r/r 258051:	.profile
r/r 258052:	.bashrc
d/d 258053:	.ssh
d/d 258092:	.cache
d/d 258111:	.ansible
r/r 258115:	.sudo_as_admin_successful
Root dir listing
$ fls -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 1550
r/r 1551:	.profile
r/r 1552:	.bashrc
d/d 258055:	.ssh
d/d 258079:	snap


CompletedProcess(args='fls -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 1550', returncode=0, stdout='r/r 1551:\t.profile\nr/r 1552:\t.bashrc\nd/d 258055:\t.ssh\nd/d 258079:\tsnap\n')

### Block 5: Allocated `/tmp` directory inventory (TOO BIASED QUESTION..)
**Question:** What allocated files exist in `/tmp`, and what do their contents reveal?

In [13]:
tmp_inode = run_command(
    ["ifind", "-o", offset_sector, "-n", "/tmp", DISK_IMAGE], 
    label="s1-23-tmp-ifind.txt", out_dir=DATA_DIR
).stdout.strip()
print(f"/tmp inode: {tmp_inode}")

tmp_allocated = run_command(
    ["fls", "-u", "-o", offset_sector, DISK_IMAGE, tmp_inode],
    label="s1-25-tmp-allocated.txt", out_dir=DATA_DIR
)
print("\nAllocated entries in /tmp:")
print(tmp_allocated.stdout.strip())


$ ifind -o 227328 -n /tmp /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01
1581
/tmp inode: 1581
$ fls -u -o 227328 /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/dumps/disk/evidence_disk.E01 1581
d/d 258067:	snap-private-tmp
d/d 258068:	.X11-unix
d/d 258069:	.ICE-unix
d/d 258070:	.XIM-unix
d/d 258071:	.font-unix
d/d 258072:	.Test-unix
r/r 74173:	__malicious_recon
r/r 74163:	__malicious_harvest

Allocated entries in /tmp:
d/d 258067:	snap-private-tmp
d/d 258068:	.X11-unix
d/d 258069:	.ICE-unix
d/d 258070:	.XIM-unix
d/d 258071:	.font-unix
d/d 258072:	.Test-unix
r/r 74173:	__malicious_recon
r/r 74163:	__malicious_harvest


**Scenario-informed selection:** From the inventory above, we investigate the known ground-truth markers `__malicious_harvest` and `__malicious_recon`. *A Better approach would have been, pivoting from the strings found in the lib*

### Block 6: Staged credential and recon evaluation
**Question:** Do the staged files contain stolen credentials or reconnaissance data?

In [ ]:
import re
tmp_entries = {}
for line in tmp_allocated.stdout.splitlines():
    if not line.startswith("r/"): continue
    parts = line.split(None, 2)
    if len(parts) >= 3:
        inode = parts[1].rstrip(":")
        name = parts[2].strip()
        tmp_entries[name] = inode

if "__malicious_harvest" in tmp_entries:
    harvest_inode = tmp_entries["__malicious_harvest"]
    harvest_content = run_command(
        ["icat", "-o", offset_sector, DISK_IMAGE, harvest_inode],
        label="s1-27-harvest-content.txt", out_dir=DATA_DIR
    )
    print(f"__malicious_harvest (inode {harvest_inode}) content begins:\n{harvest_content.stdout[:100]}...")
    
    shadow_inode = run_command(
        ["ifind", "-o", offset_sector, "-n", "/etc/shadow", DISK_IMAGE],
        label="s1-30-shadow-ifind.txt", out_dir=DATA_DIR
    ).stdout.strip()
    shadow_content = run_command(
        ["icat", "-o", offset_sector, DISK_IMAGE, shadow_inode],
        label="s1-32-shadow-content.txt", out_dir=DATA_DIR
    )
    if harvest_content.stdout == shadow_content.stdout:
        print("\n__malicious_harvest is byte-identical to /etc/shadow.")

if "__malicious_recon" in tmp_entries:
    recon_inode = tmp_entries["__malicious_recon"]
    recon_content = run_command(
        ["icat", "-o", offset_sector, DISK_IMAGE, recon_inode],
        label="s1-29-recon-content.txt", out_dir=DATA_DIR
    )
    print(f"\n__malicious_recon (inode {recon_inode}) content:\n{recon_content.stdout.strip()}")


### Block 7: Section 1 review checkpoint and findings display
The direct observations, locators, and limitations are compiled in `section1-disk-observations.md`.

In [ ]:
FINDINGS_DIR = INVESTIGATION_ROOT / "findings"
FINDINGS_DIR.mkdir(parents=True, exist_ok=True)
section1_notes = FINDINGS_DIR / "section1-disk-observations.md"
if section1_notes.is_file():
    print(section1_notes.read_text(encoding="utf-8"))
else:
    print("Observation record not found.")


## 2. Establish the surrounding activity

**Question:** what local artifacts and records explain the surrounding activity?

Retained starting point: enumerate `/tmp` without selecting scenario marker names.
Content examination, relevant accounts/history, auth/service logs, wtmp/btmp,
and the initial Plaso examination are still to be built one section at a time.
Plaso extraction must preserve parser/extraction information and original record
locators. Native log checks may verify the same underlying records, not add
independent evidence. Investigate within a justified, documented time window.

In [ ]:
tmp_inode = run_command(
    ["ifind", "-o", offset_sector, "-n", "/tmp", DISK_IMAGE],
    label="s2-ifind-tmp.txt", out_dir=DATA_DIR
).stdout.strip()
print("/tmp inode:", tmp_inode)


**Interpretation — pending.** Select candidates from the displayed evidence,
then inspect only justified content/metadata. Do not infer malice from a name
or live hiding from offline visibility. Add the initial sourced chronology here.

## 3. Follow the compromise into memory

**Not implemented.** Use the observed library/path and service context to examine
process mappings, ancestry, credentials, command lines, and sockets. Resolve the
matching symbols first. History or retained-content examination depends on the
actual process type and available structures. Avoid scenario-supplied PIDs/ports.

Output: supported runtime observations and new pivots, not automatic malware labels.

## 4. Revisit disk and missing artifacts

**Question:** what can metadata, ext4 journal reconstruction and content recovery
establish about missing/deleted material?

Deletion recovery is fully in scope. Begin with deleted-entry enumeration, then
follow the bounded journal/carving test plan in [ai/RULES.md](../../ai/RULES.md).
An empty listing does not end the investigation. Keep content, metadata-only,
bounded negative and tool-failure outcomes distinct, with preserved raw locators.
Validate candidates and document any known-input assistance before assigning
claim support or counting validated artifacts. Integrate ordinary tool commands
with the existing helper; do not build a custom recovery engine.


In [ ]:
# Deleted File Listing -- recursive fls restricted to DELETED entries only
# (-r recurse, -d deleted-only), scoped to the /tmp inode from Section 1.
fls_deleted_tmp = run_command(
    ["fls", "-o", offset_sector, "-r", "-d", "-p", str(DISK_IMAGE), tmp_inode],
    label="s4-fls-deleted-tmp.txt", out_dir=DATA_DIR
)
print("deleted directory entries under /tmp (fls -r -d):")
print(fls_deleted_tmp.stdout or "(empty -- no deleted entry listed)")


**Interpretation — pending.** A deleted name, metadata, and recovered content are
different observations. Empty output is not proof that recovery is impossible.

## Deferred - advanced tools mount and uac

### Block 2: Mounting the ewf image

Once the partition is identified, we can via `ewfmount` mount it so it can be accessible like a classical attached disk. 
This operation could be avoided because low-level tools like FTK can work with raw images.. but more advanced tool (chkrootkit, ext4magic, ecc..) need an actual accessible partition in order to work

In [ ]:
import os
from investigation_utils import is_mounted

MNT_DIR = INVESTIGATION_ROOT / "mnt"
EWF_MNT_DIR = INVESTIGATION_ROOT/"ewf"

# Create the needed dirs in order to mount correctly the ewf image
for dir in [MNT_DIR, EWF_MNT_DIR]:
    if not is_mounted(dir):
        dir.mkdir(parents=True, exist_ok=True)

# 3. Mount the E01 container locally
if is_mounted(EWF_MNT_DIR):
    print(f"[*] E01 container already mounted at {EWF_MNT_DIR}. Skipping.")
else:
    print(f"[*] Mounting E01 container to {EWF_MNT_DIR}...")
    run_command(
        ["sudo","ewfmount", DISK_IMAGE, EWF_MNT_DIR],
        label="s1-05-ewfmount",
        out_dir=OUT_DIR,
        check=True,
    )
raw_image_path = EWF_MNT_DIR / "ewf1"

# 2. Mount Ext4 Filesystem (Only if not already mounted)
if is_mounted(MNT_DIR):
    print(f"[*] Ext4 filesystem already mounted at {MNT_DIR}. Skipping.")
else:
    assert (
        offset_sector is not None
    ), f"Could not find an Ext4 partition in {DISK_IMAGE}"
    offset_bytes = offset_sector * 512
    df_options = f"ro,loop,noload,noexec,nodev,nosuid,offset={offset_bytes}"

    print(f"[*] Mounting Ext4 filesystem at offset {offset_bytes} to {MNT_DIR}...")
    run_command(
        ["sudo","mount", "-o", df_options, raw_image_path, MNT_DIR],
        label="s1-06-mount-ext4",
        out_dir=OUT_DIR,
        check=True,
    )

# 5. Verify the mount
print("\n--- Mount Verification ---")
run_command(["df", "-hT", MNT_DIR], verbose=True)

print("\n --- Root Listing ---")
run_command(f"ls -la {MNT_DIR} | head -n 10");

[*] E01 container already mounted at /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/ewf. Skipping.
[*] Ext4 filesystem already mounted at /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/mnt. Skipping.

--- Mount Verification ---
$ df -hT /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/mnt

Filesystem     Type  Size  Used Avail Use% Mounted on
/dev/loop0     ext4  9,6G  1,7G  7,9G  18% /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/mnt

 --- Root Listing ---
$ ls -la /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/mnt | head -n 10

total 88
drwxr-xr-x 19 root root  4096 set 13 22:44 .
drwxrwsr-x  9 anto kvm   4096 set 17 11:03 ..
lrwxrwxrwx  1 root root     7 mag 15 12:50 bin -> usr/bin
drwxr-xr-x  4 root root  4096 mag 15 12:55 boot
drwxr-xr-x  4 root root  

### Block 3: First disk analysis in order to check for rootkit infection via `uac`

In [ ]:
import os
from pathlib import Path

# Define output directory
UAC_OUT_DIR = INVESTIGATION_ROOT / "uac_output"
UAC_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the current notebook directory so we can return to it
notebook_dir = os.getcwd()

try:
    # 1. Temporarily change into the UAC directory
    os.chdir("/opt/uac")
    print("[*] Working directory temporarily changed to /opt/uac")

    # 2. Run UAC (forcing absolute paths for everything)
    print(f"[*] Running UAC offline collection on {MNT_DIR}...")
    uac_results = run_command(
        [
            "sudo",
            "/opt/uac/uac",
            "-p",
            "offline",
            "-m",
            str(MNT_DIR.absolute()),
            str(UAC_OUT_DIR.absolute()),
        ],
        label="s2-02-uac-offline",
        out_dir=OUT_DIR.absolute(),
        check=True,
    )

finally:
    # 3. Always return to the original notebook directory
    os.chdir(notebook_dir)
    print(f"[*] Returned to notebook directory: {notebook_dir}")

print(f"\n[*] UAC collection complete. Results saved to: {UAC_OUT_DIR}")

[*] Running UAC offline collection on /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/mnt...
$ sudo /usr/local/bin/uac -p offline -m /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/mnt /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/uac_output

Required files not found. Please ensure you are running uac from the extracted directory.


CalledProcessError: Command 'sudo /usr/local/bin/uac -p offline -m /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/mnt /home/anto/linux-multisource-dfir-lab/shared/experiments/father-u22-20260913-01/investigation/uac_output' returned non-zero exit status 1.

## 5. Assemble the incident chronology

**Not implemented. Plaso is required.** Revisit its events using the observed
paths, accounts, and times. Combine selected original disk/log events with
relevant RAM observations in one sourced chronology. Preserve timestamp meaning,
precision, unknowns, and conflicts. Do not inject scenario times to fill gaps.

Columns: time/interval | observation | original evidence locator | interpretation/limit.

## 6. Locked result tables

**Implementation pending; methodology approved on 2026-09-10.**
[ai/RULES.md](../../ai/RULES.md) is the immutable specification. Its
values remain fictional and must never be copied into this run's results.

Produce: (1) Chronology; (2) Claim Coverage & Source Support;
(3) Multi-Source Contribution; (4) Footprint Inventory.

Use a documented GT claim list with exact predicates and record references.
Manually assign S/P/U/N/A and combined conclusions from observed evidence;
retain locators, underlying origins and missing elements. Calculate the locked
counts with simple Python, preserving source independence and deduplication.
Recovery outcomes belong in these same tables. Metric selection is closed.

Compare later runs only with fixed treatment and claim definitions; re-examine
all observations rather than copying verdicts or interpreting one run as a
general distribution effect.


## 7. Validate against the controlled scenario

**Not implemented.** Once the reconstruction is written, compare it with the
frozen run's command log and input identities. Disclose prior knowledge and
assisted checks. Identify agreements, discrepancies, and unresolved actions.
Ground truth is an experimental reference, not another forensic evidence source.